## Lab 4: Deploy to Production - Use AgentCore Runtime with Observability

### Overview

In Lab 3 we scaled our Customer Support Agent by centralizing tools through AgentCore Gateway with secure authentication. Now we'll complete the production journey by deploying our agent to AgentCore Runtime with comprehensive observability. This will transform our prototype into a production-ready system that can handle real-world traffic with full monitoring and automatic scaling.

[Amazon Bedrock AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html) is a secure, fully managed runtime that empowers organizations to deploy and scale AI agents in production, regardless of framework, protocol, or model choice. It provides enterprise-grade reliability, automatic scaling, and comprehensive monitoring capabilities.

**Workshop Journey:**

- **Lab 1 (Done):** Create Agent Prototype - Built a functional customer support agent
- **Lab 2 (Done):** Enhance with Memory - Added conversation context and personalization
- **Lab 3 (Done):** Scale with Gateway & Identity - Shared tools across agents securely
- **Lab 4 (Current):** Deploy to Production - Used AgentCore Runtime with observability
- **Lab 5:** Evaluate Agent Performance - Monitor quality with online evaluations
- **Lab 6:** Build User Interface - Create a customer-facing application

### Why AgentCore Runtime & Production Deployment Matter

Current State (Lab 1-3): Agent runs locally with centralized tools but faces production challenges:

- Agent runs locally in a single session
- No comprehensive monitoring or debugging capabilities
- Cannot handle multiple concurrent users reliably

After this lab, we will have a production-ready agent infrastructure with:

- Serverless auto-scaling to handle variable demand
- Comprehensive observability with traces, metrics, and logging
- Enterprise reliability with automatic error recovery
- Secure deployment with proper access controls
- Easy management through AWS console and APIs and support for real-world production workloads.


### Adding comprehensive observability with AgentCore Observability

Additionally, AgentCore Runtime integrates seamlessly with [AgentCore Observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html) to provide full visibility into your agent's behavior in production. AgentCore Observability automatically captures traces, metrics, and logs from your agent interactions, tool usage, and memory access patterns. In this lab we will see how AgentCore Runtime integrates with CloudWatch GenAI Observability to provide comprehensive monitoring and debugging capabilities.

For request tracing, AgentCore Observability captures the complete conversation flow including tool invocations, memory retrievals, and model interactions. For performance monitoring, it tracks response times, success rates, and resource utilization to help optimize your agent's performance.

During the observability flow, AgentCore Runtime automatically instruments your agent code and sends telemetry data to CloudWatch. You can then use CloudWatch dashboards and GenAI Observability features to analyze patterns, identify bottlenecks, and troubleshoot issues in real-time.

### Architecture for Lab 4
<div style="text-align:left"> 
    <img src="images/architecture_lab4_runtime.png" width="75%"/> 
</div>

*Agent now runs in AgentCore Runtime with full observability through CloudWatch, serving production traffic with auto-scaling and comprehensive monitoring. Memory and Gateway integrations from previous labs remain fully functional in the production environment.*

### Key Features

- **Serverless Agent Deployment:** Transform your local agent into a scalable production service using AgentCore Runtime with minimal code changes
- **Comprehensive Observability:** Full request tracing, performance metrics, and debugging capabilities through CloudWatch GenAI Observability


**IMPORTANT!!!**: You MUST enable [CloudWatch Transaction Search](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/Enable-TransactionSearch.html) to be able to see AgentCore Observability traces in CloudWatch. (This is done for you by the agentcore starter toolkit)


### Step 1: Import Required Libraries

In [1]:
# Import required libraries
import boto3
from lab_helpers.lab2_memory import create_or_get_memory_resource

create_or_get_memory_resource()  # Just in case the memory lab wasn't executed

'CustomerSupportMemory-cNJQnMEmoS'

### Step 2: Preparing Your Agent for AgentCore Runtime

#### Creating the Runtime-Ready Agent

Let's first define the necessary AgentCore Runtime components via Python SDK within our previous local agent implementation.

Observe the `#### AGENTCORE RUNTIME - LINE i ####` comments below to see where is the relevant deployment code added. You'll find 4 such lines that prepare the runtime-ready agent:

1. Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
2. Initialize the App with `app = BedrockAgentCoreApp()`
3. Decorate our invocation function with `@app.entrypoint`
4. Let AgentCore Runtime control the execution with `app.run()`


In [2]:
%%writefile ./lab_helpers/lab4_runtime.py
from bedrock_agentcore.runtime import (
    BedrockAgentCoreApp,
)  #### AGENTCORE RUNTIME - LINE 1 ####
from strands import Agent
from strands.models import BedrockModel
from scripts.utils import get_ssm_parameter
from retrying import retry
from lab_helpers.lab1_strands_agent import (
    get_return_policy,
    get_product_info,
    get_technical_support,
    SYSTEM_PROMPT,
    MODEL_ID,
)

from lab_helpers.lab2_memory import (
    CustomerSupportMemoryHooks,
    memory_client,
    ACTOR_ID,
    SESSION_ID,
)

# Retry logic for handling throttling and service unavailable errors
def should_retry(exception):
    error_str = str(exception).lower()
    return 'throttl' in error_str or 'serviceunavailable' in error_str

@retry(retry_on_exception=should_retry,
       wait_fixed=5000,
       stop_max_delay=30*1000)
def call_with_retry(f):
    return f()

# Lab1 import: Create the Bedrock model
model = BedrockModel(model_id=MODEL_ID)

# Lab2 import : Initialize memory via hooks
memory_id = get_ssm_parameter("/app/customersupport/agentcore/memory_id")
memory_hooks = CustomerSupportMemoryHooks(
    memory_id, memory_client, ACTOR_ID, SESSION_ID
)

# Create the agent with all customer support tools
agent = Agent(
    model=model,
    tools=[get_return_policy, get_product_info, get_technical_support],
    system_prompt=SYSTEM_PROMPT,
    hooks=[memory_hooks],
)

# Initialize the AgentCore Runtime App
app = BedrockAgentCoreApp()  #### AGENTCORE RUNTIME - LINE 2 ####


@app.entrypoint  #### AGENTCORE RUNTIME - LINE 3 ####
def invoke(payload):
    """AgentCore Runtime entrypoint function"""
    user_input = payload.get("prompt", "")

    # Invoke the agent with retry logic
    response = call_with_retry(lambda: agent(user_input))
    return response.message["content"][0]["text"]

if __name__ == "__main__":
    app.run()  #### AGENTCORE RUNTIME - LINE 4 ####

Writing ./lab_helpers/lab4_runtime.py


#### What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

- Creates an HTTP server that listens on port 8080
- Implements the required `/invocations` endpoint for processing requests
- Implements the `/ping` endpoint for health checks
- Handles proper content types and response formats
- Manages error handling according to AWS standards


### Step 3: Deploying to AgentCore Runtime

Now let's deploy our agent to AgentCore Runtime using the [AgentCore Starter Toolkit](https://github.com/aws/bedrock-agentcore-starter-toolkit).

#### Configure the Secure Runtime Deployment (AgentCore Runtime + AgentCore Identity)

First we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we will create and a requirements file. We will also configure the identity authorization using an Amazon Cognito user pool and we will configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code

<div style="text-align:left"> 
    <img src="images/configure.png" width="75%"/> 
</div>

**Note**: The Cognito access_token is valid for 2 hours only. If the access_token expires you can vend another access_token by using the `reauthenticate_user` method.


In [3]:
from lab_helpers.utils import setup_cognito_user_pool, reauthenticate_user

print("Setting up Amazon Cognito user pool...")
cognito_config = (
    setup_cognito_user_pool()
)  # You'll get your bearer token from this output cell.
print("Cognito setup completed ✓")

Setting up Amazon Cognito user pool...
{'UserPoolId': 'us-east-1_zsiswuZ7T', 'ClientName': 'MCPServerPoolClient', 'ClientId': '3c27ie0mcpvdtmn81c8nbc0qov', 'ClientSecret': '4u02t6f1ovk951gcnoi9b5q2is9prn8pboibh7tbt79mjacsjdk', 'LastModifiedDate': datetime.datetime(2026, 3, 18, 11, 5, 13, 817000, tzinfo=tzlocal()), 'CreationDate': datetime.datetime(2026, 3, 18, 11, 5, 13, 817000, tzinfo=tzlocal()), 'RefreshTokenValidity': 30, 'TokenValidityUnits': {}, 'ExplicitAuthFlows': ['ALLOW_USER_PASSWORD_AUTH', 'ALLOW_USER_SRP_AUTH', 'ALLOW_REFRESH_TOKEN_AUTH'], 'AllowedOAuthFlowsUserPoolClient': False, 'EnableTokenRevocation': True, 'EnablePropagateAdditionalUserContextData': False, 'AuthSessionValidity': 3}
Pool id: us-east-1_zsiswuZ7T
Discovery URL: https://cognito-idp.us-east-1.amazonaws.com/us-east-1_zsiswuZ7T/.well-known/openid-configuration
Client ID: 3c27ie0mcpvdtmn81c8nbc0qov
Bearer Token: eyJraWQiOiJTdVpwTU03ZzBWRzFNQ0p5Q0F5U01FSEwwTFQ4d2F6RkF2VWtnWmtuXC9iaz0iLCJhbGciOiJSUzI1NiJ9.eyJzdWI

In [4]:
from bedrock_agentcore_starter_toolkit import Runtime
from lab_helpers.utils import create_agentcore_runtime_execution_role

# Initialize the runtime toolkit
boto_session = boto3.session.Session()
region = boto_session.region_name

execution_role_arn = create_agentcore_runtime_execution_role()

agentcore_runtime = Runtime()

# Configure the deployment
response = agentcore_runtime.configure(
    entrypoint="lab_helpers/lab4_runtime.py",
    execution_role=execution_role_arn,
    auto_create_ecr=True,
    requirements_file="requirements-agent.txt",
    region=region,
    agent_name="customer_support_agent",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
)

print("Configuration completed:", response)

/home/participant/.local/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


✅ Created IAM role: CustomerSupportAssistantBedrockAgentCoreRole-us-east-1
Role ARN: arn:aws:iam::410571135192:role/CustomerSupportAssistantBedrockAgentCoreRole-us-east-1
✅ Created policy: CustomerSupportAssistantBedrockAgentCorePolicy-us-east-1
✅ Attached policy to role
Policy ARN: arn:aws:iam::410571135192:policy/CustomerSupportAssistantBedrockAgentCorePolicy-us-east-1


Entrypoint parsed: file=/workshop/agentcore/lab_helpers/lab4_runtime.py, bedrock_agentcore_name=lab4_runtime
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: customer_support_agent


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


📄 Generated Dockerfile: /workshop/agentcore/Dockerfile

Generated .dockerignore: /workshop/agentcore/.dockerignore
Setting 'customer_support_agent' as default agent
Bedrock AgentCore configured: /workshop/agentcore/.bedrock_agentcore.yaml


Configuration completed: config_path=PosixPath('/workshop/agentcore/.bedrock_agentcore.yaml') dockerfile_path=PosixPath('/workshop/agentcore/Dockerfile') dockerignore_path=PosixPath('/workshop/agentcore/.dockerignore') runtime='None' runtime_type=None region='us-east-1' account_id='410571135192' execution_role='arn:aws:iam::410571135192:role/CustomerSupportAssistantBedrockAgentCoreRole-us-east-1' ecr_repository=None auto_create_ecr=True s3_path=None auto_create_s3=False memory_id=None network_mode='PUBLIC' network_subnets=None network_security_groups=None network_vpc_id=None


#### Launch the Agent

Now let's launch our agent to AgentCore Runtime. This will create an AWS CodeBuild pipeline, the Amazon ECR repository and the AgentCore Runtime components.

<div style="text-align:left"> 
    <img src="images/launch.png" width="100%"/> 
</div>

In [5]:
# Launch the agent (this will build and deploy the container)
from lab_helpers.utils import put_ssm_parameter

launch_result = agentcore_runtime.launch()
print("Launch completed:", launch_result.agent_arn)

agent_arn = put_ssm_parameter(
    "/app/customersupport/agentcore/runtime_arn", launch_result.agent_arn
)

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'customer_support_agent' to account 410571135192 (us-east-1)
Generated image tag: 20260318-110538-117
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: customer_support_agent
ECR repository available: 410571135192.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-customer_support_agent
Using execution role from config: arn:aws:iam::410571135192:role/CustomerSupportAssistantBedrockAgentCoreRole-us-east-1
Preparing CodeBuild project and uploading source...


Repository doesn't exist, creating new ECR repository: bedrock-agentcore-customer_support_agent


Getting or creating CodeBuild execution role for agent: customer_support_agent
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-aa96c3a063
CodeBuild role doesn't exist, creating new role: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-aa96c3a063
Creating IAM role: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-aa96c3a063
✓ Role created: arn:aws:iam::410571135192:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-aa96c3a063
Attaching inline policy: CodeBuildExecutionPolicy to role: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-aa96c3a063
✓ Policy attached: CodeBuildExecutionPolicy
Waiting for IAM role propagation...
CodeBuild execution role creation complete: arn:aws:iam::410571135192:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-aa96c3a063
Created S3 bucket: bedrock-agentcore-codebuild-sources-410571135192-us-east-1
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: customer_support_agent/source.zip
Created CodeBuild project: bedrock-agentcore-cu

Launch completed: arn:aws:bedrock-agentcore:us-east-1:410571135192:runtime/customer_support_agent-VcNz3b40aI


#### Check Deployment Status

Let's wait for the deployment to complete:


In [6]:
import time

# Wait for the agent to be ready
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]

end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    print(f"Waiting for deployment... Current status: {status}")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]

print(f"Final status: {status}")

Retrieved Bedrock AgentCore status for: customer_support_agent


Final status: READY


### Step 4: Invoking Your Deployed Agent

Now that our agent is deployed and ready, let's test it with some queries. We invoke the agent with the right authorization token type. In out case it'll be Cognito access token. Copy the access token from the cell above

<div style="text-align:left"> 
    <img src="images/invoke.png" width="100%"/> 
</div>

#### Using the AgentCore Starter Toolkit

We can validate that the agent works using the AgentCore Starter Toolkit for invocation. The starter toolkit can automatically create a session id for us to query our agent. Alternatively, you can also pass the session id as a parameter during invocation. For demonstration purpose, we will create our own session id.

In [7]:
import uuid

# Create a session ID for demonstrating session continuity
session_id = uuid.uuid4()

# Test different customer support scenarios
user_query = "My Iphone is not connecting with the Bluetooth. What should I do?"

bearer_token = reauthenticate_user(
    cognito_config.get("client_id"), 
    cognito_config.get("client_secret")
)

response = agentcore_runtime.invoke(
    {"prompt": user_query}, 
    bearer_token=bearer_token,
    session_id=str(session_id)
)
response['response']

Using JWT authentication


'"Based on the technical support information, here are the steps to troubleshoot your iPhone Bluetooth connectivity issue:\\n\\n## **iPhone Bluetooth Troubleshooting Steps:**\\n\\n### **1. Basic Bluetooth Fixes:**\\n- **Restart Bluetooth**: Turn Bluetooth off and on again in Settings > Bluetooth\\n- **Clear Bluetooth cache**: Go to Settings > General > Reset > Reset Network Settings (this will also reset Wi-Fi passwords)\\n- **Restart both devices**: Restart your iPhone and the device you\'re trying to connect to\\n\\n### **2. Re-pairing Process:**\\n- **Remove existing pairing**: In Settings > Bluetooth, tap the \\"i\\" next to the problematic device and select \\"Forget This Device\\"\\n- **Put accessory in pairing mode**: Follow the specific pairing instructions for your device\\n- **Re-pair from scratch**: Look for the device in your Bluetooth settings and pair again\\n\\n### **3. Additional Checks:**\\n- **Verify compatibility**: Ensure your device is compatible with your iPhone m

#### Invoking the agent with session continuity

Since we are using AgentCore Runtime, we can easily continue our conversation with the same session id.

In [8]:
user_query = "I've turned my Bluetooth on and off but it still does not work"
response = agentcore_runtime.invoke(
    {"prompt": user_query}, 
    bearer_token=bearer_token,
    session_id=str(session_id)
)
response

Using JWT authentication


{'response': '"Since the basic Bluetooth restart didn\'t resolve the issue, let\'s try more advanced troubleshooting steps:\\n\\n## **Advanced Bluetooth Troubleshooting:**\\n\\n### **1. Reset Network Settings (Most Effective):**\\n- Go to **Settings > General > Transfer or Reset iPhone > Reset > Reset Network Settings**\\n- This will clear all Bluetooth cache and network configurations\\n- **Note**: You\'ll need to re-enter all Wi-Fi passwords\\n\\n### **2. Check for System Updates:**\\n- Go to **Settings > General > Software Update**\\n- Install any available iOS updates as they often fix Bluetooth bugs\\n\\n### **3. Force Restart Your iPhone:**\\nFor iPhone 14 specifically:\\n- Press and quickly release the **Volume Up** button\\n- Press and quickly release the **Volume Down** button  \\n- Press and hold the **Side** button until you see the Apple logo\\n\\n### **4. Check Device Compatibility:**\\n- What specific device are you trying to connect to? (headphones, speaker, car, etc.)\\

#### Invoking the agent with a new user
In the example below we have not mentioned the Iphone device in the second query, but our agent still has the context of it. This is due to the AgentCore Runtime session continuity. The agent won't know the context for a new user.

In [9]:
# Creating a new session ID for demonstrating new customer
session_id2 = uuid.uuid4()

user_query = "Still not working. What is going on?"
response = agentcore_runtime.invoke(
    {"prompt": user_query}, 
    bearer_token=bearer_token,
    session_id=str(session_id2)
)
response

Using JWT authentication


{'response': '"Based on our general troubleshooting resources, here are some common steps to try when devices aren\'t working properly:\\n\\n## First, let\'s try these basic troubleshooting steps:\\n\\n**Power Issues:**\\n- Check all power cable connections\\n- Try a different power outlet\\n- Look for any physical damage to the device\\n- If it\'s a laptop or mobile device, ensure the battery isn\'t completely drained\\n\\n**Performance Issues:**\\n- Close any unnecessary applications\\n- Check if you have enough storage space available\\n- Try restarting the device completely\\n- Make sure your device software/firmware is up to date\\n\\n**Connectivity Problems:**\\n- If it\'s a network issue, try restarting your router and the device\\n- Check that Wi-Fi passwords are correct\\n- For Bluetooth issues, try unpairing and re-pairing devices\\n\\nTo give you more specific help, could you tell me:\\n1. What type of device is having the problem?\\n2. What exactly isn\'t working - is it no

In this case our agent does not have the context anymore and needs more information. 

And it is all it takes to have a secure and scalable endpoint for our Agent with no need to manage all the underlying infrastructure!

### Step 5: AgentCore Observability

[AgentCore Observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html) provides monitoring and tracing capabilities for AI agents using Amazon OpenTelemetry Python Instrumentation and Amazon CloudWatch GenAI Observability.

#### Agents

Default AgentCore Runtime configuration allows for logging our agent's traces on CloudWatch by means of **AgentCore Observability**. These traces can be seen on the AWS CloudWatch GenAI Observability dashboard. Navigate to Cloudwatch &rarr; GenAI Observability &rarr; Bedrock AgentCore.

![Agents Overview on CloudWatch](images/observability_agents.png)

#### Sessions

The Sessions view shows the list of all the sessions associated with all agents in your account.

![sessions](images/sessions_lab5_observability.png)

#### Traces

Trace view lists all traces from your agents in this account. To work with traces:

- Choose Filter traces to search for specific traces.
- Sort by column name to organize results.
- Under Actions, select Logs Insights to refine your search by querying across your log and span data or select Export selected traces to export.

![traces](images/traces_lab4_observability.png)


### Additional Challenge
Amazon Bedrock AgentCore Runtime has a powerful capability that extends beyond just hosting AI agents - it can also host MCP servers. This means you can deploy custom tools and services as MCP servers that can be consumed by various AI agents and applications.

Your task is to build a simple MCP server and deploy it on AgentCore Runtime, use this [documantation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-mcp.html#runtime-mcp-create-server) 

### Congratulations! 🎉

You have successfully completed **Lab 4: Deploy to Production - Use AgentCore Runtime with Observability!**

Here is what you accomplished:

##### Production-Ready Deployment:

- Prepared your agent for production with minimal code changes (only 4 lines added)
- Validated proper session isolation between different customers
- Confirmed session continuity + memory persistence and context awareness per session

##### Enterprise-Grade Security & Identity:

- Implemented secure authentication using Cognito integration with JWT tokens
- Configured proper IAM roles and execution permissions for production workloads
- Established identity-based access control for secure agent invocation

##### Comprehensive Observability:

- Enabled AgentCore Observability for full request tracing across all customer sessions
- Configured CloudWatch GenAI Observability dashboard monitoring

##### Current Limitations (We'll fix these next!):

- **Developer Focused Interaction** - Agent accessible via SDK/API calls but no user-friendly web interface
- **Manual Session Management** - Requires programmatic session creation rather than intuitive user experience

##### Next Up [Lab 5: Evaluate Agent Performance →](lab-05-agentcore-evals.ipynb)
In Lab 5, you'll configure online evaluation to automatically assess agent performance in real-time!
